# 01 番茄叶片八分类：数据准备

本研究统一使用七种番茄病害与健康叶片八个类别。PlantVillage `color` 是模型训练数据来源，PlantDoc 是独立外部评价来源。**PlantDoc 原目录 `train` 与 `test` 只表示来源，不参与本研究训练、验证、模型选择或超参数调整。**

从第一格依次运行。数据准备会打开图片检查有效性、计算完全重复哈希，并按种子 42 生成约 70% / 15% / 15% 的 PlantVillage 分层划分；原图不会被修改。清单与质量报告写入 `data/splits/` 和 `data/metadata/`。本 Notebook 不训练模型。


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请在项目根目录或 notebooks 目录打开本 Notebook。")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config, load_labels

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")
CLASS_NAMES = [item["label"] for item in load_labels(PROJECT_ROOT)]
assert len(CLASS_NAMES) == 8 and len(set(CLASS_NAMES)) == 8
assert [item["label_index"] for item in LABELS["classes"]] == list(range(8))
print("项目目录：", PROJECT_ROOT)
print("类别数量：", len(CLASS_NAMES), "；随机种子：", BASE["seed"])


## 1. 核对固定标签与原始数量

类别编号 0—7 在训练、验证、推理和测试中保持相同。只使用指定八类；target spot、spider mites 等其他原始类别仍保留在磁盘，但不加入清单。

`tomato_leaf → 健康`、`tomato_leaf_yellow_virus → 黄化曲叶病毒病` 是本研究的**暂定映射**。现有源说明不足以证明它们逐图语义一致，原始文件名还有潜在错标提示；不得把相似名称或模型预测当作病理学确认。


In [ ]:
from src.data import dataset_inventory

display(pd.DataFrame(LABELS["classes"]))
inventory = dataset_inventory(PROJECT_ROOT)
display(pd.DataFrame(inventory))


## 2. 扫描质量并生成可复现划分

以下单元实际执行扫描。明确损坏图片和完全重复文件从研究清单中排除，原始图像、原始标注和来源元数据保留。存在跨类别完全重复时，按质量报告记录的规则处置，不猜测真实类别。

可可靠恢复的官方来源组优先用于分组划分。官方分组覆盖有限，花叶病毒类缺少官方分组匹配；无法恢复的样本使用保守的独立标识。**哈希无重复、来源组无交叉仍不能证明所有生物叶片或近重复照片均已隔离。** 不要求对数万张图逐张人工审核。

首次扫描需要几分钟。准备完成后保持清单固定，再进行 02—05；研究过程中修改数据或配置后，应重新准备并重新训练。


In [ ]:
from src.data import prepare_manifests

summary = prepare_manifests(
    PROJECT_ROOT,
    load_config(PROJECT_ROOT / "configs" / "labels.yaml"),
    seed=42,
)
print("当前数据指纹：", summary["fingerprint"])
print("各清单样本数：")
display(pd.DataFrame([{"manifest": name, "samples": count}
                      for name, count in summary["counts"].items()]))
print("各类样本数与划分：")
display(pd.DataFrame(summary["per_class"]))
print("质量与来源分组报告：")
display(summary["quality"])


## 3. 查看清洗后类别分布与样本

图中类别使用固定编号，中文名称见上方标签表。比较样本数和类别不平衡，后续重点报告 Macro-F1 与逐类 Recall。PlantDoc 来源数量应分别保留；合并外部集合经过整体去重，不把源目录文件数直接当作有效评价数。


In [ ]:
from src.data import read_manifest
from PIL import Image

manifest_names = ["pv_train", "pv_val", "pv_test", "plantdoc_external_train",
                  "plantdoc_external_test", "plantdoc_external_all"]
rows_by_split = {
    name: read_manifest(PROJECT_ROOT / "data" / "splits" / f"{name}.csv")
    for name in manifest_names
}
counts = pd.DataFrame({name: pd.Series([int(row["label_index"]) for row in rows]).value_counts()
                       for name, rows in rows_by_split.items()}).reindex(range(8), fill_value=0).fillna(0).astype(int)
counts.index.name = "label_index"
display(counts)
ax = counts.plot.bar(figsize=(11, 4), rot=0)
ax.set(xlabel="Class index", ylabel="Images", title="Eight-class manifest distribution")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for index, axis in enumerate(axes.flat):
    sample = next(row for row in rows_by_split["pv_train"] if int(row["label_index"]) == index)
    with Image.open(PROJECT_ROOT / sample["image_path"]) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(f"Class {index}")
    axis.axis("off")
plt.tight_layout()
plt.show()


## 4. 检查数据加载

本单元只检查所有清单的首批图片能否读取，不开展模型训练，也不利用测试样本选择参数。数据质量扫描已经逐图验证文件可打开；完整验收记录由项目检查流程另行保存。


In [ ]:
from src.data import build_transforms, make_loader

data_config = BASE["data"]
transform = build_transforms("none", **{key: data_config[key]
                            for key in ("image_size", "mean", "std") if key in data_config})
for name in manifest_names:
    loader = make_loader(PROJECT_ROOT / "data" / "splits" / f"{name}.csv", PROJECT_ROOT,
                         transform, batch_size=BASE["training"]["batch_size"],
                         shuffle=False, num_workers=0, seed=42)
    images, labels = next(iter(loader))
    assert images.shape[1:] == (3, data_config["image_size"], data_config["image_size"])
    assert labels.min().item() >= 0 and labels.max().item() < 8
    print(name, "：总数", len(loader.dataset), "首批图像", tuple(images.shape),
          "标签范围", (labels.min().item(), labels.max().item()))


## 结论记录（运行后填写）

- 八类清洗前后数量、类别不平衡：待填写。
- 损坏、完全重复、标签冲突及处置：依据本次质量报告填写。
- 来源组覆盖、回退样本数量与剩余泄漏风险：待填写。
- PlantDoc 两项暂定映射的限制：待填写。
- 清单生成日期与版本：待填写。下一步打开 **02**，先查看计划，再开启训练。
